In [19]:
import subprocess
import sys
import random as rd
import webbrowser

def instalar(package): #Instala o pacote necessário caso ele não esteja instalado
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

try:
    import pandas as pd
except ImportError:
    instalar('pandas')
    import pandas as pd

In [20]:
#Cria um backup para garantir que a coleção não seja sobreescrita acidentalmente
try: 
    backup = pd.read_csv('save.csv')
except:
    backup = pd.DataFrame(columns=['set', 'id', 'name', 'grade', 'clan', 'type', 'rarity', 'qtt']) 
backup.to_csv('save_backup.csv', index=False)

In [21]:
#CSV que armazena o nome dos sets e seus links
direct = pd.read_csv('packs\direct.csv')

In [22]:
def escolhe_pacote(name):
    view = input('Deseja ver as cartas desse pacote na Wiki?(s/n): ')
    if view == 's':
        link = direct[direct['set'] == name.upper()]['link'].values[0]
        webbrowser.open(link)
    else:
        data = pd.read_csv(f'packs\{name}.csv')
        for index, row in data.iterrows(): 
            print(f'0{row["id"]} {row["name"]} - {row["grade"]} - {row["clan"]} - {row["type"]} - {row["rarity"]}')


In [23]:
def gerar_link_wiki(nome_carta):
    # A wiki usa underscores no lugar de espaços
    nome_formatado = nome_carta.replace(' ', '_')
    return f'https://cardfight.fandom.com/wiki/{nome_formatado}'

In [24]:
def ver_colecao(nome, web=False):
    try:
        save = pd.read_csv(f'{nome}.csv')
        if web == False:
            for index, row in save.iterrows():
                print(f'0{row["id"]} {row["name"]} - {row["grade"]} - {row["clan"]} - {row["type"]} - {row["rarity"]} - Qtt: {row["qtt"]}')
        else:
            print(f'Abrindo a Wiki de: {len(save)} cartas...')
            for index, row in save.iterrows():
                link = gerar_link_wiki(row['name'])
                webbrowser.open(link) 
    except:
        print('Você ainda não tem nenhuma carta na coleção! Abra alguns pacotes para começar a colecionar!')

In [25]:
""" def web_colecao(nome):
    try:
        save = pd.read_csv(f'{nome}.csv')
        for index, row in save.iterrows():
            link = gerar_link_wiki(row['name'])
            print(f'Abrindo Wiki de: {row["name"]}...')
            webbrowser.open(link) # Isso abre o navegador automaticamente
    except:
        print('Erro ao carregar coleção.') """

' def web_colecao(nome):\n    try:\n        save = pd.read_csv(f\'{nome}.csv\')\n        for index, row in save.iterrows():\n            link = gerar_link_wiki(row[\'name\'])\n            print(f\'Abrindo Wiki de: {row["name"]}...\')\n            webbrowser.open(link) # Isso abre o navegador automaticamente\n    except:\n        print(\'Erro ao carregar coleção.\') '

In [26]:
def filtra_colecao(tipo, filtro, data = 'save'):
    if data == 'save':
        save = pd.read_csv(f'{data}.csv')
    else:
        save = data.copy()
    
    if (tipo == 'type') and (filtro.lower() == 'trigger'):
        save = save[(save['type'].str.lower()) != 'normal unit']
    elif (tipo == 'grade'):
        save = save[(save['grade'] == int(filtro))]
    else:
        save = save[(save[tipo].str.lower()) == filtro]

    if save.empty:
        print(f'Nenhuma carta encontrada para o filtro: {tipo} = {filtro}')
    else:
        for index, row in save.iterrows():
            print(f'0{row["id"]} {row["name"]} - {row["grade"]} - {row["clan"]} - {row["type"]} - {row["rarity"]} - Qtt: {row["qtt"]}')
    return save    

In [27]:
def atualizar_save(box, pacote):
    #Quero colocar os valores do pacote na box e se houver repetidos, qtt += 1
    
    for index, row in pacote.iterrows():
        card = box[(box['set'] == row['set']) & (box['id'] == row['id'])]
        if not card.empty:
            box.loc[card.index, 'qtt'] += 1
        else:
            new_card = row.to_dict()
            new_card['qtt'] = 1
            #box não tem append, então vou criar um novo dataframe com a nova linha e concatenar com a box
            new_card_df = pd.DataFrame([new_card])
            box = pd.concat([box, new_card_df], ignore_index=True)

    return box

In [28]:
def rodar_pacote(name):
    data = pd.read_csv(f'packs\{name}.csv')
    pacote = []
    

    commons = data[data['rarity'] == 'C']
    c_tiradas = commons.sample(n=4).to_dict('records')
    pacote.extend(c_tiradas)

    luck = rd.randint(1, 100)

    if luck <= 70:
        raridade = 'R'
    elif luck <= 90:
        raridade = 'RR'
    elif luck <= 99:
        raridade = 'RRR'
    else:
        raridade = 'SP'

    raras = data[data['rarity'] == raridade]
    r_tirada = raras.sample(n=1).to_dict('records')
    pacote.extend(r_tirada)

    pacote = pd.DataFrame(pacote)
    
    return pacote

In [29]:
def rodar_box(name, qtt):
    box = pd.DataFrame(columns=['set', 'id', 'name', 'grade', 'clan', 'type', 'rarity', 'qtt'])
    
    for i in range(qtt):
        pacote = rodar_pacote(name)
        box = atualizar_save(box, pacote)
    
    try:
        save = pd.read_csv('save.csv')
    except:
        save = pd.DataFrame(columns=['set', 'id', 'name', 'grade', 'clan', 'type', 'rarity', 'qtt'])

    box.sort_values(by=['id'], inplace=True)
    box.to_csv('last_box.csv', index=False)

    save = atualizar_save(save, box)
    save.sort_values(by=['set', 'id'], inplace=True)
    save.to_csv('save.csv', index=False)
    
    return box